# Генерация Терры

Модель `ldm_v4` дообучена с двумя условиями:

- **кто**: другая кошка или Терра;
- **тип**: кластер признаков DINOv2. Кластеры в основном про окрас и породу, Терра в кластере длинношерстных колорпойнтов
  вместе с ~330 похожими кошками.

При генерации работает classifier-free guidance (CFG): выход модели с условиями отодвигается от выхода без условий
с силой `W`. Больше `W` = больше похоже на Терру, но меньше разнообразие и выше риск пересыщенных цветов.
Поверх можно включить autoguidance слабой моделью (`GUIDE`).

30 фото Терры отложены и в обучении не участвовали (`terra_holdout.json`). Последняя ячейка показывает
для каждой картинки ближайшее обучающее фото и отмечает копии.

**Внимание:** финальный снимок `ckpt/last.ckpt` (шаг 25 000) в основном копирует обучающие фото: около 60% сэмплов
это копии против 3% у настоящих новых фото. Какой снимок брать, покажут метрики в `runs/eval/terra_v4/`.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import AutoencoderKL
from PIL import Image, ImageOps

from dataset import read_manifest
from guidance import CFG, Guided
from models import UNet
from train import VPSDE, sample

RUN_DIR = Path('/home/CCP/semenchuk.aa/DiffusionLearn/runs/ldm_v4')
DATA = Path('/home/CCP/semenchuk.aa/DiffusionLearn_data/pretrain_v1')
DEVICE = 'cuda'

cfg = json.loads((RUN_DIR / 'config.json').read_text())
WHO_NULL, K = cfg['cond_sizes']        # значения «пусто» у условий: кто = 2, тип = 100
TERRA_TYPE = cfg['terra_type']         # основной кластер Терры
print(f"шагов {cfg['steps']}, Терра {cfg['terra_frac']:.0%} примеров, кластеров {K}, тип Терры {TERRA_TYPE}, scale {cfg['scale']:.5f}")

vae = AutoencoderKL.from_pretrained(cfg['vae'], local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)
sde = VPSDE(pred=cfg['pred'])
bf16 = lambda: torch.autocast('cuda', dtype=torch.bfloat16)


def load_ema(path):
    """EMA-веса из ckpt/last.ckpt или ema_*.pt любого запуска, конфигурация из его config.json. Возвращает модель и шаг."""
    path = Path(path)
    run = path.parent.parent if path.parent.name == 'ckpt' else path.parent
    c = json.loads((run / 'config.json').read_text())
    ck = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    sd = ck['ema'] if 'ema' in ck else {k[4:]: v for k, v in ck['state_dict'].items() if k.startswith('ema.')}
    m = UNet(img_size=64, in_ch=4, base=c['base'], mults=[int(v) for v in c['mults'].split(',')], num_res=c['num_res'],
             attn_res=[int(v) for v in c['attn_res'].split(',')], dropout=c['dropout'], norm_groups=32,
             head_ch=c.get('head_ch') or None, qk_norm=c.get('qk_norm', False), cond_sizes=c.get('cond_sizes', []))
    m.load_state_dict(sd)
    return m.to(DEVICE).eval().requires_grad_(False), ck.get('step', ck.get('global_step'))


@torch.no_grad()
def decode(z, chunk=8):
    """Латенты модели (B, 4, 64, 64) -> картинки uint8 (B, 3, 512, 512) на CPU."""
    out = []
    for i in range(0, len(z), chunk):
        x = vae.decode(z[i:i + chunk].to(DEVICE).float() / cfg['scale']).sample
        out.append(((x.clamp(-1, 1) + 1) * 127.5).round().byte().cpu())
    return torch.cat(out)


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) для imshow."""
    return img.permute(1, 2, 0).numpy()


@torch.no_grad()
def generate_terra(model, n, seed=0, w=2.0, use_type=True, steps=250, guide=None, w_ag=1.5, chunk=16):
    """
    n картинок Терры через SDE. Возвращает латенты (n, 4, 64, 64) на CPU.
      w        : сила CFG от ответа без условий, 1 = без CFG
      use_type : добавить условие «тип Терры» (кластер колорпойнтов), иначе только «кто = Терра»
      guide    : слабая модель для autoguidance с теми же условиями, w_ag ее сила
    Один и тот же seed дает одни и те же стартовые шумы при любых w, поэтому варианты можно сравнивать.
    """
    y = torch.tensor([1, TERRA_TYPE if use_type else K])
    y0 = torch.tensor([WHO_NULL, K])
    net = CFG(model, y, y0, w)
    if guide is not None:
        net = Guided(net, CFG(guide, y, y0, w), w_ag)
    x0 = torch.randn(n, 4, 64, 64, generator=torch.Generator().manual_seed(seed))
    out = []
    for i in range(0, n, chunk):
        torch.manual_seed(seed * 1_000_003 + i)                     # шум SDE тоже повторяется
        out.append(sample(net, sde, x0[i:i + chunk].to(DEVICE), steps=steps, rho=1.0, method='sde',
                          amp_ctx=bf16, clip=False).cpu())
    return torch.cat(out)

## Генерация

Картинки сохраняются в `runs/ldm_v4/generated_terra/<имя>/`, имя собирается из параметров.

In [ ]:
# ------------------------------------------------------------- параметры
CKPT = 'ckpt/last.ckpt'      # или 'ema_0015000.pt': ранние снимки меньше похожи на Терру, но меньше и копируют
W = 2.0                      # сила CFG: 1 = без нее, 2-3 обычно хорошо, выше пересыщает
USE_TYPE = True              # условие «тип Терры»: сильнее держит окрас
GUIDE = None                 # autoguidance, например '/home/CCP/semenchuk.aa/DiffusionLearn/runs/ldm_v3/ckpt/last.ckpt'
W_AG = 1.5
SEED = 0
BATCH = 16
STEPS = 250
COLS = 4

# ------------------------------------------------------------- генерация
model, step = load_ema(RUN_DIR / CKPT)
guide = load_ema(GUIDE)[0] if GUIDE else None
lat = generate_terra(model, BATCH, seed=SEED, w=W, use_type=USE_TYPE, steps=STEPS, guide=guide, w_ag=W_AG)
imgs = decode(lat)
del guide
torch.cuda.empty_cache()

name = f'step{step:07d}_w{W}' + ('_type' if USE_TYPE else '') + (f'_ag-{Path(GUIDE).parts[-3]}-w{W_AG}' if GUIDE else '') + f'_seed{SEED}'
out_dir = RUN_DIR / 'generated_terra' / name
out_dir.mkdir(parents=True, exist_ok=True)
for i, im in enumerate(imgs):
    Image.fromarray(to_np(im)).save(out_dir / f'{i:03d}.png')
print(f'{BATCH} картинок сохранено в {out_dir}')

rows = math.ceil(BATCH / COLS)
fig, axs = plt.subplots(rows, COLS, figsize=(4.5 * COLS, 4.7 * rows), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for i, im in enumerate(imgs):
    axs.flat[i].imshow(to_np(im))
    axs.flat[i].set_title(str(i))
plt.suptitle(name, fontsize=15)
plt.tight_layout()
plt.show()

## Сила CFG

Одни и те же шумы, строка = `W`. Видно, как растет похожесть на Терру и падает разнообразие.

In [ ]:
W_LIST = [1.0, 1.5, 2.0, 3.0, 4.0]
N_CMP = 6

rows = [decode(generate_terra(model, N_CMP, seed=SEED, w=w, use_type=USE_TYPE, steps=STEPS)) for w in W_LIST]
fig, axs = plt.subplots(len(W_LIST), N_CMP, figsize=(2.8 * N_CMP, 2.9 * len(W_LIST)), squeeze=False)
for r, (w, ims) in enumerate(zip(W_LIST, rows)):
    for c in range(N_CMP):
        axs[r, c].imshow(to_np(ims[c]))
        axs[r, c].set_xticks([])
        axs[r, c].set_yticks([])
    axs[r, 0].set_ylabel(f'W = {w}', fontsize=13)
plt.suptitle(f'step {step}, ' + ('кто + тип' if USE_TYPE else 'только кто'), fontsize=14)
plt.tight_layout()
plt.show()

## Новая это Терра или копия

Как в `tools/eval_terra.py`: каждое обучающее фото Терры раскладывается на кропы по правилу обучения
(бокс внутри, несколько размеров и сдвигов, отражение) в 64x64, и для картинки ищется ближайший кроп.
d1 = расстояние до ближайшего фото, d2 = до второго ближайшего другого фото, `ratio = d1 / d2`.
Копия близка к одному фото намного сильнее, чем к остальным: `ratio < 0.7`, подпись красная.
У 30 отложенных настоящих фото Терры так выходит около 3%.

Первый запуск ячейки около минуты: строятся кропы 254 обучающих фото.

In [ ]:
from tools.eval_terra import copy_ratio, copy_refs

N_SHOW = 8

if 'refs' not in globals():
    hold = set(json.loads((DATA / 'terra_holdout.json').read_text()))
    train_rows = [r for r in read_manifest(DATA / 'manifest.jsonl') if r['source'] == 'terra' and r['path'] not in hold]
    refs, owner = copy_refs(DATA, train_rows)

x = imgs[:N_SHOW].float() / 127.5 - 1
ratio, nn = copy_ratio(x, refs, owner, len(train_rows))
ratio, nn = ratio.cpu(), nn.cpu()
print(f'ratio по картинкам: ' + ', '.join(f'{v:.2f}' for v in ratio.tolist())
      + f'; копий {(ratio < 0.7).sum().item()} из {N_SHOW}')

fig, axs = plt.subplots(math.ceil(N_SHOW / 2), 4, figsize=(16, 4.2 * math.ceil(N_SHOW / 2)), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for j in range(N_SHOW):
    r, c = divmod(j, 2)
    real = ImageOps.fit(ImageOps.exif_transpose(Image.open(DATA / train_rows[nn[j]]['path'])).convert('RGB'), (512, 512))
    axs[r, 2 * c].imshow(to_np(imgs[j]))
    axs[r, 2 * c].set_title(f'{j}: ratio {ratio[j]:.2f}' + (', копия' if ratio[j] < 0.7 else ''),
                            color='red' if ratio[j] < 0.7 else 'black')
    axs[r, 2 * c + 1].imshow(real)
    axs[r, 2 * c + 1].set_title('ближайшее обучающее фото')
plt.tight_layout()
plt.show()